# 🛰️ Live Satellite Tracker Notebook: Overview
## **Real-Time Positions → Ground Tracks → Live Swaths → ROI Countdown**

This notebook gives an **n2yo-style live view** centred on your ROI: where each satellite is now, its ground track for the last 45 and next 90 minutes, its current instrument swath, the day/night terminator, and a countdown to each satellite's next imaging overpass of the ROI, with clocks in **UTC** and the **ROI's local time**.

---

### 🚀 Two Ways to Watch
| Option | Runs where | Best for |
| :--- | :--- | :--- |
| **`live_tracker`** (ipyleaflet) | Inside Jupyter, on Esri World Imagery | Working sessions — updates every 2 s while the notebook stays usable. |
| **`export_live_tracker_html`** | Any browser, no Python | Sharing, GitHub Pages, a second screen. Orbits are propagated in the browser (satellite.js / SGP4). |

> ⚠️ **Note**: ipyleaflet widgets need JupyterLab ≥ 3 or Notebook ≥ 7 with widgets enabled (included in `env.base.yml`). GitHub cannot render live widgets.

## ⚙️ Workspace Configuration, ROI & Orbital Elements
> ### **Same ROI and TLE cache as the prediction notebook**

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "Tools")

from overpass_helper_utility import (
    load_roi,
    resolve_roi_timezone,
    fetch_tles,
    predict_overpasses,
    next_overpasses,
    satellite_positions,
    live_tracker,
    export_live_tracker_html,
)

ROI_GDF = load_roi(Path("DATA") / "SHP" / "Port_Phillip_Bay_Extent_EPSG_32755.shp", name="Port_Phillip_Bay")
ROI_TZ = resolve_roi_timezone(ROI_GDF)
TLES = fetch_tles(tle_dir=Path("DATA") / "TLE")

# Upcoming overpasses feed the countdown column
PASSES = predict_overpasses(ROI_GDF, TLES, days=16, timezone_name=ROI_TZ, verbose=False)
display(next_overpasses(PASSES))

## 📍 Current Positions (Snapshot)
A one-off table of sub-satellite position, altitude, ground speed and direction — the same numbers the live panel refreshes.

In [ ]:
display(satellite_positions(TLES))

## 📡 Live Tracker (ipyleaflet)
> ### **Async Update Loop → Map Layers → Telemetry Panel**

---

### 📂 Map Layers
| Layer | Meaning |
| :--- | :--- |
| **Dot** | Current sub-satellite point. |
| **Dashed / solid line** | Ground track, last 45 min / next 90 min (`track_minutes`). |
| **Shaded strip** | Instrument swath for ±2.5 min around now. When it covers the red ROI, the ROI is being imaged. |
| **Dark overlay** | Night side of the terminator. |

Updates run as an `asyncio` task on the kernel's event loop, so other cells keep working. Use the **Live** toggle, or run the stop cell.

In [ ]:
TRACKER = live_tracker(TLES, ROI_GDF, passes=PASSES, refresh_s=2, track_minutes=(45, 90), zoom=3)
TRACKER

In [ ]:
# Stop live updates (restart with TRACKER.start())
TRACKER.stop()

## 🌐 Standalone Live-Tracker Page
> ### **Embedded TLEs + Predictions → Browser-Side SGP4 → Shareable HTML**

Writes `OUTPUT/live_tracker.html`, a single page that keeps tracking in real time in any browser. It embeds the current TLEs, the ROI, the predicted overpasses and a Natural Earth coastline, and loads only `satellite.js` from a CDN. Drag to pan, scroll to zoom, switch between **World** and **ROI region**, and click a satellite to hide or show it.

> 💡 **Tip**: The embedded TLEs age. The page footer shows TLE age per satellite; re-export after refreshing TLEs.

In [ ]:
html_path = export_live_tracker_html(TLES, ROI_GDF, Path("OUTPUT") / "live_tracker.html", passes=PASSES)